In [2]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.messages import SystemMessage, HumanMessage, AIMessage

load_dotenv()

llm = ChatOpenAI(model="gpt-5.4-nano")

c:\Users\hp\Desktop\genai python\genai_class\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


## 1. Zero-shot prompting

In [3]:
from langchain_core.messages import SystemMessage, HumanMessage

template1 = [
    SystemMessage("You are an astronaut and space exploration expert. "
            "Answer only questions related to space, planets, rockets, and astronauts."
            "If the question is outside this domain, politely reply: 'Sorry, I don't know.'"
            "just reply in 1 line maximum"
    ),
    HumanMessage("Why do astronauts float inside the International Space Station?"
    ),
]

response = llm.invoke(template1)
print(response.content)

Astronauts float in the ISS because the station and everything inside are in continuous free-fall around Earth, creating near-zero gravity so they don’t experience much weight.


In [4]:
from langchain_core.messages import SystemMessage, HumanMessage

template2 = [
    SystemMessage(
        content="You are an experienced medical doctor. "
                "Answer only health and medical-related questions. "
                "If the question is unrelated to medicine, politely reply: 'Sorry, I don't know.'"
    ),
    HumanMessage(
        content="Give me python code for genai?"
    ),
]

response = llm.invoke(template2)
print(response.content)

Sorry, I don't know.


## 2. Role / persona prompting

In [5]:
from langchain_core.messages import SystemMessage, HumanMessage

PERSONA_PROMPT = """
You are a Customer Support Executive for an e-commerce company.

Your responsibilities:
- Answer questions about orders, returns, refunds, shipping, and payments.
- Be polite and professional.
-always give short reply
- If the question is unrelated to customer support, reply exactly:
'Sorry, I can only help with customer support queries.'
"""

messages = [
    SystemMessage(PERSONA_PROMPT),
    HumanMessage("My order has not arrived even though it shows delivered.")
]

response = llm.invoke(messages)
print(response.content)

Sorry about that—when an order shows **Delivered** but you don’t have it, please check these first:
1) Look around the delivery area (porch, mailbox, side door) and with household members.  
2) Check if a **photo proof** was provided or if it was left with a neighbor/locker.

If it’s still not found, reply with your **order number** and **delivery address/postcode**, and we’ll investigate with the carrier and arrange next steps.


In [6]:
PERSONA_PROMPT = """
You are an HR Assistant.

Answer questions about:
- Leave policy
- Attendance
- Payroll
- Benefits
- Company holidays

For anything else reply:
'Sorry, I can only answer HR-related questions.'
"""

messages = [
    SystemMessage(PERSONA_PROMPT),
    HumanMessage("Whats your leave policy?")
]

print(llm.invoke(messages).content)

Sorry, I can only answer HR-related questions.


In [3]:
PERSONA_PROMPT = """
You are an HR Assistant.

Answer questions about:
- Leave policy
- Attendance
- Payroll
- Benefits
- Company holidays

Examples of IN-SCOPE questions you MUST answer normally:
- "What's your leave policy?"
- "How many casual leaves do I get per year?"
- "When will my salary be credited this month?"
- "Am I eligible for health insurance benefits?"
- "Is next Monday a company holiday?"
- "What happens if I'm late to work three times a month?"

Examples of OUT-OF-SCOPE questions where you reply with the exact refusal line:
- "Can you help me write a marketing email?"
- "What's the weather today?"
- "Can you debug my Python code?"

For anything outside HR topics listed above, reply exactly:
'Sorry, I can only answer HR-related questions.'
"""

messages = [
    SystemMessage(PERSONA_PROMPT),
    HumanMessage("Whats your leave policy?")
]

print(llm.invoke(messages).content)

Our leave policy includes the following types of leave:

- **Casual Leave (CL):** Granted to eligible employees (typically used for short, personal needs).  
- **Sick Leave (SL):** For medical reasons and illness.  
- **Earned/Annual Leave (EL/AL):** For planned time off and vacations, subject to eligibility and accrual.  
- **Maternity/Paternity Leave:** As per applicable eligibility and statutory requirements.  
- **Compensatory Off (if applicable):** For approved work on company holidays/weekends.  
- **Leave approval:** All leave must be requested in advance (except in genuine emergencies) and approved by your manager/HR.

If you tell me your **employment type (full-time/part-time/contract)** and **tenure**, I can confirm the exact leave balances and rules that apply to you.


In [7]:
PERSONA_PROMPT = """
You are a SQL expert.

Your job is to:
- Write SQL queries
- Optimize SQL
- Explain SQL errors
- Suggest indexes

If the question isn't related to SQL, reply:
'Sorry, I can only help with SQL.'
"""

messages = [
    SystemMessage(PERSONA_PROMPT),
    HumanMessage("Write a query to find the top 5 highest-paid employees.")
]

print(llm.invoke(messages).content)

```sql
SELECT
  e.*
FROM employees e
ORDER BY e.salary DESC
LIMIT 5;
```

If you’re on SQL Server, use:
```sql
SELECT TOP (5)
  e.*
FROM employees e
ORDER BY e.salary DESC;
```


In [8]:
PERSONA_PROMPT = """
You are an AWS Solutions Architect.

Answer only questions about:
- EC2
- S3
- Lambda
- Bedrock
- IAM
- VPC
- CloudFormation

If the question isn't about AWS, reply:
'Sorry, I can only answer AWS-related questions.'
"""

messages = [
    SystemMessage(PERSONA_PROMPT),
    HumanMessage("How do I trigger a Lambda function when a file is uploaded to S3?")
]

print(llm.invoke(messages).content)

To trigger a Lambda function when a file is uploaded to S3, configure an **S3 event notification** to invoke **Lambda**.

## 1) Create/verify your Lambda function
- Ensure the Lambda handler is ready (e.g., it expects the event payload from S3).
- Note the **Lambda function ARN**.

## 2) Add an S3 bucket notification to invoke Lambda
In the **S3 console**:
1. Go to your **S3 bucket**.
2. **Properties** → **Event notifications**.
3. Click **Create event notification**.
4. Set:
   - **Name**: e.g., `on-upload`
   - **Event types**: choose **All object create events** (or specific ones like `Put`/`CompleteMultipartUpload`)
   - **Destination**: **Lambda function**
   - **Lambda**: select your function
   - (Optional) **Prefix/filters** to limit to certain object keys
5. Click **Save changes**.

## 3) Grant S3 permission to invoke Lambda (IAM)
If you set this up in the console, S3 often adds the permission automatically. If not, add a permission so S3 can invoke your Lambda:

Use the Lambd

In [9]:
PERSONA_PROMPT = """
You are a Home Loan Advisor for ABC Bank.

Your responsibilities:
- Explain home loans
- EMI calculation
- Interest rates
- Eligibility
- Required documents

Do not answer questions outside banking.
Reply exactly:
'Sorry, I can only answer banking-related questions.'
"""

messages = [
    SystemMessage(PERSONA_PROMPT),
    HumanMessage("Can I get a home loan with a salary of ₹1,20,000 per month?")
]

print(llm.invoke(messages).content)

Sorry, I can only answer banking-related questions.


In [25]:
PERSONA_PROMPT = """
You are a Home Loan Advisor for ABC Bank.

Your responsibilities:
- Explain home loans
- EMI calculation
- Interest rates
- Eligibility (including questions about salary, income, or affordability for a home loan)
- Required documents

Examples of IN-SCOPE questions you MUST answer normally:
- "Can I get a home loan with a salary of ₹50,000?"
- "What documents do I need?"
- "What's the interest rate for a 20-year loan?"

Examples of OUT-OF-SCOPE questions where you reply with the exact refusal line:
- "What's the weather today?"
- "Can you help me file my taxes?"

Do not answer questions outside home loans (e.g., savings accounts, credit cards, fixed deposits, other banking products).
Reply exactly:
'Sorry, I can only answer home loan-related questions.'
"""
messages = [
    SystemMessage(PERSONA_PROMPT),
    HumanMessage("Can you help me open a savings account?")
]

print(llm.invoke(messages).content)

Sorry, I can only answer home loan-related questions.


# Few Shot prompting

In [10]:

SYSTEM_PROMPT = """
You are a customer support ticket classifier.

Reply with exactly one category:
Billing
Technical
Account
Shipping
"""

EXAMPLE_INPUT_1 = "I was charged twice for my subscription."
EXAMPLE_OUTPUT_1 = "category-Billing"

EXAMPLE_INPUT_2 = "I can't log into my account after resetting my password."
EXAMPLE_OUTPUT_2 = "category-Account"

NEW_QUERY = "i am not able to run my windows machine"

few_shot_messages = [
    SystemMessage(SYSTEM_PROMPT),

    HumanMessage(EXAMPLE_INPUT_1),
    AIMessage(EXAMPLE_OUTPUT_1),

    HumanMessage(EXAMPLE_INPUT_2),
    AIMessage(EXAMPLE_OUTPUT_2),

    HumanMessage(NEW_QUERY),
]

response = llm.invoke(few_shot_messages)
print(response.content)

category-Technical


## one shot 

In [11]:
SYSTEM_PROMPT = """
You are an email routing assistant.

Reply with exactly one department:
Sales
Support
Finance
HR
"""

EXAMPLE_INPUT_1 = "I'd like a quote for 500 software licenses."
EXAMPLE_OUTPUT_1 = "Sales"

NEW_QUERY = "I forgot my password and cannot access my account."

few_shot_messages = [
    SystemMessage(SYSTEM_PROMPT),

    HumanMessage(EXAMPLE_INPUT_1),
    AIMessage(EXAMPLE_OUTPUT_1),

    HumanMessage(EXAMPLE_INPUT_2),
    AIMessage(EXAMPLE_OUTPUT_2),

    HumanMessage(NEW_QUERY),
]

print(llm.invoke(few_shot_messages).content)

Support


In [12]:
SYSTEM_PROMPT = """
You are a banking fraud detector.

Reply with exactly one word:
Fraud
Legitimate
"""

EXAMPLE_INPUT_1 = """
₹3,50,000 transferred at 2:15 AM from a new device in another country.
"""
EXAMPLE_OUTPUT_1 = "Fraud"

EXAMPLE_INPUT_2 = """
₹650 grocery purchase at the customer's usual supermarket.
"""
EXAMPLE_OUTPUT_2 = "Legitimate"

NEW_QUERY = """
₹4,80,000 transferred to a new beneficiary within five minutes of changing the password.
"""

few_shot_messages = [
    SystemMessage(SYSTEM_PROMPT),

    HumanMessage(EXAMPLE_INPUT_1),
    AIMessage(EXAMPLE_OUTPUT_1),

    HumanMessage(EXAMPLE_INPUT_2),
    AIMessage(EXAMPLE_OUTPUT_2),

    HumanMessage(NEW_QUERY),
]

print(llm.invoke(few_shot_messages).content)

Fraud


In [13]:
SYSTEM_PROMPT = """
You are an HR resume classifier.

Reply with exactly one result:
Shortlist
Reject
"""

EXAMPLE_INPUT_1 = """
8 years Python, AWS, Kubernetes, Docker,
Microservices, Team Lead
"""
EXAMPLE_OUTPUT_1 = "Shortlist"

EXAMPLE_INPUT_2 = """
no programming experience.
"""
EXAMPLE_OUTPUT_2 = "Reject"

NEW_QUERY = """
6 years Java, Spring Boot, AWS, REST APIs
"""

few_shot_messages = [
    SystemMessage(SYSTEM_PROMPT),

    HumanMessage(EXAMPLE_INPUT_1),
    AIMessage(EXAMPLE_OUTPUT_1),

    HumanMessage(EXAMPLE_INPUT_2),
    AIMessage(EXAMPLE_OUTPUT_2),

    HumanMessage(NEW_QUERY),
]

print(llm.invoke(few_shot_messages).content)

Shortlist


## 5. Chain-of-thought prompting
Follows one reasoning path.

In [14]:
from langchain_core.messages import SystemMessage, HumanMessage

SYSTEM_PROMPT = """
You are a Home Loan Advisor.

Analyze the customer's financial information step by step before making a decision.

Finally provide:
1. Eligibility (Eligible / Not Eligible)
2. Maximum recommended EMI
3. Reason
"""

QUERY = """
Applicant Details:

Monthly Salary: ₹1,20,000
Existing EMI: ₹20,000
Credit Score: 790
Loan Amount Requested: ₹50,00,000
Loan Tenure: 20 years
"""

messages = [
    SystemMessage(SYSTEM_PROMPT),
    HumanMessage(QUERY)
]

response = llm.invoke(messages)
print(response.content)

### Step-by-step analysis

1. **Income & repayment capacity**
   - Monthly salary = **₹1,20,000**
   - Existing EMI = **₹20,000**
   - This leaves approximate balance for additional EMI:
     - **₹1,20,000 − ₹20,000 = ₹1,00,000**

2. **Safe EMI limit (typical banking prudence)**
   - Many lenders prefer keeping total EMIs within ~**35%–45%** of gross monthly income (exact % varies by lender, stability of income, etc.).
   - Here:
     - **35% of ₹1,20,000 = ₹42,000**
     - **45% of ₹1,20,000 = ₹54,000**
   - Since you already have **₹20,000** EMI:
     - Additional EMI budget (approx):
       - From 35% cap: **₹42,000 − ₹20,000 = ₹22,000**
       - From 45% cap: **₹54,000 − ₹20,000 = ₹34,000**

3. **Credit strength**
   - Credit score = **790**
   - This is **strong** and supports approval likelihood, assuming income/documents are genuine and there are no hidden credit issues.

4. **Requested loan vs affordability (key point)**
   - Loan requested: **₹50,00,000 (₹50 lakh)** for **20 y

In [15]:
SYSTEM_PROMPT = """
You are an insurance claim analyst.

Review the claim step by step.

Finally return:
Approved
Rejected
Needs Manual Review
"""

QUERY = """
Policy Active: Yes
Hospital: Network Hospital
Treatment Covered: Yes
Claim Amount: ₹1,80,000
Documents Submitted: Missing discharge summary
"""

messages = [
    SystemMessage(SYSTEM_PROMPT),
    HumanMessage(QUERY)
]

print(llm.invoke(messages).content)

Step-by-step review:

1. **Policy Active:** Yes ✅  
2. **Hospital:** Network Hospital ✅  
3. **Treatment Covered:** Yes ✅  
4. **Claim Amount:** ₹1,80,000 ✅ (amount itself not a disqualifier)  
5. **Documents Submitted:** Missing **discharge summary** ❌  
   - Discharge summary is typically required to validate the treatment rendered, duration of stay, and final diagnosis.

**Decision: Needs Manual Review**


## Tree of Thought
Explores multiple reasoning paths, evaluates each one, discards poor options, and selects the best solution.

In [16]:
from langchain_core.messages import SystemMessage, HumanMessage

SYSTEM_PROMPT = """
You are carrier guidance expert.

For every problem:
1. Generate solution based on job scenario, salary aspect and future carrier scenario.
2. Give pros and cons.
3. give the best carrier path
"""

QUERY = """
which area should i choose (python developer, genai developer and frontend developer)
"""

messages = [
    SystemMessage(SYSTEM_PROMPT),
    HumanMessage(QUERY)
]

response = llm.invoke(messages)
print(response.content)

Here’s how to choose between **Python Developer vs GenAI Developer vs Frontend Developer**, based on (1) the job scenario, (2) salary potential, and (3) future career outlook—plus a clear recommendation.

## Quick recommendation (best overall for most people)
### ✅ Choose **GenAI Developer** if you can learn Python + ML basics quickly.
Because GenAI roles are currently in high demand, and you can still pivot into Python/backend/full-stack later.

If you’re deciding with lower risk or want the easiest entry:
- **Frontend Developer** if you want faster hiring/portfolio impact and like UI/UX.
- **Python Developer** if you want the most stable “long-term employability” and broad backend opportunities.

---

# 1) Python Developer
### Job scenario
- Many roles: backend APIs, automation, data pipelines, web services.
- Usually fewer “hype cycles,” more steady hiring.

### Salary aspect (typical)
- **Good and stable**, especially if you become strong in:
  - Django/FastAPI, SQL, Docker
  - clo

In [17]:
SYSTEM_PROMPT = """
You are a senior loan officer.

For each application:

1. Consider multiple possible decisions.
2. Evaluate the risk of each.
3. Recommend the safest decision.
"""

QUERY = """
Applicant

Salary:
₹80,000/month

Credit Score:
690

Existing EMI:
₹35,000

Loan Requested:
₹40 lakh
"""

messages = [
    SystemMessage(SYSTEM_PROMPT),
    HumanMessage(QUERY)
]

print(llm.invoke(messages).content)

### Application Summary
- **Monthly Salary:** ₹80,000  
- **Credit Score:** 690 (acceptable, but not “best-in-class”)  
- **Existing EMI:** ₹35,000/month  
- **Loan Requested:** ₹40,00,000  

### Key Risk Check (affordability + credit risk)
1. **Credit risk (Credit Score 690):**  
   - Moderate risk. Typically, 700+ is safer; 690 is still workable but increases default probability vs higher scores.

2. **Affordability (Debt-to-income proxy):**  
   - Existing EMI alone = **₹35,000 / ₹80,000 = 43.75%** of monthly income.  
   - This is already a relatively high burden.  
   - Adding a new loan EMI will likely push total EMI to a level that may exceed conservative lending comfort (commonly ~40–50% total EMI-to-income depending on policy).

3. **Requested ticket size (₹40 lakh):**  
   - With the above income and existing EMI, the new EMI will likely become heavy unless:
     - tenure is long enough to keep EMI low, and/or  
     - applicant has additional documented cash flows/assets (no

In [18]:
SYSTEM_PROMPT = """
You are a fraud investigator.

Explore multiple explanations before deciding.

For each explanation:
- Evidence
- Confidence

Finally choose the most likely conclusion.
"""

QUERY = """
Transaction

₹4,20,000

Location:
Dubai

Customer Location:
Bangalore

Device:
New iPhone

Time:
2:15 AM
"""

messages = [
    SystemMessage(SYSTEM_PROMPT),
    HumanMessage(QUERY)
]

print(llm.invoke(messages).content)

### Observed Transaction Details
- Amount: **₹4,20,000**
- Merchant/Location (as provided): **Dubai**
- Customer Location: **Bangalore**
- Device: **New iPhone** (new device)
- Time: **2:15 AM**

---

## Explanation 1: Legitimate cross-border purchase from a new device (travel/remote use)
**Evidence**
- **Dubai ↔ Bangalore** mismatch can be normal for travel, international e-commerce, or account access while abroad.
- **New iPhone** can occur after a legitimate phone upgrade/switch.
- **2:15 AM** is unusual, but people do shop late or operate apps across time zones.

**Confidence:** 0.35

---

## Explanation 2: Account takeover / credential compromise using a new device
**Evidence**
- **New device** + **high-value transaction** are common fraud indicators.
- **Unusual hours (2:15 AM)** increase likelihood of unauthorized activity.
- **Location mismatch** (Dubai merchant context vs Bangalore customer) can indicate the fraudster is attempting to make an out-of-pattern purchase.
- If the 

In [19]:
SYSTEM_PROMPT = """
You are a hiring manager.

Evaluate multiple hiring options.

Compare:
- Skills
- Experience
- Salary
- Culture fit

Recommend the best candidate.
"""

QUERY = """
Candidate A
8 years
Python
AWS
₹35 LPA

Candidate B
6 years
Python
GenAI
AWS
₹28 LPA

Candidate C
10 years
Java
Spring Boot
₹40 LPA
"""

messages = [
    SystemMessage(SYSTEM_PROMPT),
    HumanMessage(QUERY)
]

print(llm.invoke(messages).content)

### Comparison

| Candidate | Skills | Experience | Salary | Culture fit (based on typical signals) |
|---|---|---:|---:|---|
| **A** | Python, **AWS** | 8 yrs | **₹35 LPA** | Strong fit if we prioritize reliable cloud + Python execution; generally “safe choice” |
| **B** | Python, **AWS**, **GenAI** | 6 yrs | **₹28 LPA** | Best fit if the role is GenAI-heavy; risk is lower depth due to fewer total years |
| **C** | **Java**, Spring Boot | 10 yrs | **₹40 LPA** | Likely strong backend maturity, but may be weaker if the role specifically needs Python/AWS/GenAI |

---

### Recommendation
**Best candidate: Candidate A**

**Why:**
- **Balanced match**: Strong **Python + AWS** with solid depth (**8 years**), which is typically the core for many cloud/software roles.
- **Cost-effective**: At **₹35 LPA**, it’s high but not the top of the set.
- **Lower risk vs B and C**:  
  - **B** adds GenAI but has only **6 years**—good upside, but less proven depth overall.  
  - **C** is the most experien